# 06 — CUPED Variance Reduction (Stretch)

CUPED = Controlled-experiment Using Pre-Experiment Data
Microsoft technique, now standard at Amazon, Airbnb, Netflix.

Goal: Show variance reduction on a constructed A/B test scenario.

Key insight: pre-experiment covariate (e.g., units' operating hours before the test)
correlates with the outcome — adjust it out to reduce variance and increase
statistical power without collecting more data.

Formula: Y_cuped = Y - theta * (X - E[X]) where theta = Cov(Y, X) / Var(X)

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

np.random.seed(42)

# Simulate a simple A/B test with a pre-experiment covariate
N = 2000
pre_covariate = np.random.normal(200, 40, N)  # e.g., avg operating hours before test
true_effect   = -0.05  # treatment reduces failure rate by 5 percentage points
noise         = np.random.normal(0, 0.15, N)

treatment = np.random.choice([0, 1], size=N)
# Outcome: failure rate correlated with pre-covariate AND treatment
outcome = 0.15 + true_effect * treatment + 0.0005 * (pre_covariate - 200) + noise

df = pd.DataFrame({'treatment': treatment, 'outcome': outcome, 'pre_covariate': pre_covariate})

In [ ]:
# Standard t-test
trt = df[df.treatment == 1].outcome
ctrl = df[df.treatment == 0].outcome
t_stat, p_val = stats.ttest_ind(trt, ctrl)
print(f'Standard t-test: effect = {trt.mean()-ctrl.mean():.4f}, SE = {np.std(trt.values - ctrl.mean()) / np.sqrt(len(trt)):.4f}, p = {p_val:.4f}')

In [ ]:
# CUPED adjustment
theta = np.cov(df.outcome, df.pre_covariate)[0, 1] / np.var(df.pre_covariate)
df['outcome_cuped'] = df.outcome - theta * (df.pre_covariate - df.pre_covariate.mean())

trt_c = df[df.treatment == 1].outcome_cuped
ctrl_c = df[df.treatment == 0].outcome_cuped
t_stat_c, p_val_c = stats.ttest_ind(trt_c, ctrl_c)
print(f'CUPED t-test:    effect = {trt_c.mean()-ctrl_c.mean():.4f}, SE = {np.std(trt_c.values - ctrl_c.mean()) / np.sqrt(len(trt_c)):.4f}, p = {p_val_c:.4f}')

var_reduction = (1 - df.outcome_cuped.var() / df.outcome.var()) * 100
print(f'\nVariance reduction: {var_reduction:.1f}%')
print(f'Effect: same. Variance: lower. Power: higher. No extra data needed.')